In [ ]:
import os
import yaml
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets
from tqdm import tqdm

from lsst.ts.ofc import OFC, OFCData, StateEstimator
from StarSharp import StateFactory

import matplotlib.pyplot as plt
%matplotlib widget

In [ ]:
mttcs_dir = Path(os.environ["TS_CONFIG_MTTCS_DIR"])
mtaos_dir = mttcs_dir / "MTAOS/v13/ofc/"

senspath = mtaos_dir / "sensitivity_matrix" / "lsst_sensitivity_dz_31_29_50.yaml"
with open(senspath, "r") as f:
    sens = np.array(yaml.safe_load(f))
normpath = mtaos_dir / "normalization_weights" / "range-fwhm.yaml"
with open(normpath, "r") as f:
    norm = np.array(yaml.safe_load(f))

DOF_NAMES = []
for component in ["M2", "Cam"]:
    for dof in ["dz", "dx", "dy", "rx", "ry"]:
        DOF_NAMES.append(f"{component} {dof}")
for component in ["M1M3", "M2"]:
    for dof in range(1, 21):
        DOF_NAMES.append(f"{component} B{dof}")

In [ ]:
def str_to_array(s):
    if isinstance(s, (list, np.ndarray)):
        return np.array(s)
    result = []
    for part in s.split(","):
        if "-" in part:
            start, end = map(int, part.split("-"))
            result.extend(range(start, end + 1))
        else:
            result.append(int(part))
    return np.array(result)

In [ ]:
def mtaos_mixing_matrix(use_dof="0-9,10-16,30-34", n_keep=12, noll_indices="4-19,22-26"):
    global state_estimator
    use_dof = str_to_array(use_dof)
    noll_indices = str_to_array(noll_indices)

    ofc_data = OFCData(
        'lsst',
        config_dir=mtaos_dir
    )
    new_comp_dof_idx = dict(
        m2HexPos=np.full(5, False, dtype=bool),
        camHexPos=np.full(5, False, dtype=bool),
        M1M3Bend=np.full(20, False, dtype=bool),
        M2Bend=np.full(20, False, dtype=bool),
    )
    for idof in use_dof:
        if idof < 5:
            new_comp_dof_idx["m2HexPos"][idof] = True
        elif 5 <= idof < 10:
            new_comp_dof_idx["camHexPos"][idof-5] = True
        elif 10 <= idof < 30:
            new_comp_dof_idx["M1M3Bend"][idof-10] = True
        elif 30 <= idof < 50:
            new_comp_dof_idx["M2Bend"][idof-30] = True
    ofc = OFC(ofc_data)
    ofc.ofc_data.comp_dof_idx = new_comp_dof_idx
    ofc.ofc_data.zn_selected = noll_indices
    ofc.set_truncation_index(n_keep)

    state_estimator = StateEstimator(ofc_data)
    out = []
    for imode in range(n_keep):
        vmodes = np.zeros(n_keep)
        vmodes[imode] = 1.0
        out.append(state_estimator.get_dofs_from_vmodes(vmodes))
    return np.array(out)

In [ ]:
@ipywidgets.interact_manual(
    use_dof="0-9,10-16,30-34",
    n_keep=12,
    noll_indices="4-19,22-26",
    normalize=ipywidgets.Checkbox(value=True, description="Normalize"),
)
def f(use_dof, n_keep, noll_indices, normalize):
    global mix
    use_dof = str_to_array(use_dof)
    noll_indices = str_to_array(noll_indices)
    mix = mtaos_mixing_matrix(use_dof=use_dof, n_keep=len(use_dof), noll_indices=noll_indices)
    if normalize:
        mix = mix @ np.diag(1/norm[use_dof])

    vmax = np.max(np.abs(mix))
    fig, ax = plt.subplots(ncols=1, nrows=1, figsize=(8, 6))
    ax.imshow(mix.T, origin="lower", cmap="bwr", vmin=-vmax, vmax=vmax)
    m2_hex_idx = np.where(use_dof < 5)[0]
    cam_hex_idx = np.where((use_dof >= 5) & (use_dof < 10))[0]
    m1m3_bend_idx = np.where((use_dof >= 10) & (use_dof < 30))[0]
    m2_bend_idx = np.where(use_dof >= 30)[0]
    if len(m2_hex_idx) > 0:
        if any(len(block) > 0 for block in [cam_hex_idx, m1m3_bend_idx, m2_bend_idx]):
            ax.axhline(m2_hex_idx[-1] + 0.5, color="k", alpha=0.2)
    if len(cam_hex_idx) > 0:
        if any(len(block) > 0 for block in [m1m3_bend_idx, m2_bend_idx]):
            ax.axhline(cam_hex_idx[-1] + 0.5, color="k", alpha=0.2)
    if len(m1m3_bend_idx) > 0:
        if len(m2_bend_idx) > 0:
            ax.axhline(m1m3_bend_idx[-1] + 0.5, color="k", alpha=0.2)
    ax.set_xticks(range(mix.shape[1]))
    ax.set_xticklabels(range(1, mix.shape[1] + 1))
    ax.axvspan(n_keep - 0.5, mix.shape[1] - 0.5, color="r", alpha=0.2)
    ax.set_yticks(range(mix.shape[1]))
    ax.set_yticklabels([DOF_NAMES[i] for i in use_dof])
    plt.show()

In [ ]:
@ipywidgets.interact_manual(
    use_dof="0-9,10-16,30-34",
    n_keep=12,
    noll_indices="4-19,22-26",
    normalize=ipywidgets.Checkbox(value=True, description="Normalize"),
)
def f(use_dof, n_keep, noll_indices, normalize):
    global sf
    use_dof = str_to_array(use_dof)
    ss = np.array(sens)
    ss[..., [3,4,8,9]] /= 3600  # deg to arcsec
    nn = np.array(norm)
    nn[[3,4,8,9]] *= 3600  # deg to arcsec
    sf = StateFactory(
        ss,
        nn,
        use_dof=use_dof,
        n_keep=len(use_dof),
    )
    Vh = np.array(sf.Vh)
    if normalize:
        Vh = Vh @ np.diag(1/nn[use_dof])

    vmax = np.max(Vh)
    fig, ax = plt.subplots(ncols=1, nrows=1, figsize=(8, 6))
    ax.imshow(Vh.T, origin="lower", cmap="bwr", vmin=-vmax, vmax=vmax)
    m2_hex_idx = np.where(use_dof < 5)[0]
    cam_hex_idx = np.where((use_dof >= 5) & (use_dof < 10))[0]
    m1m3_bend_idx = np.where((use_dof >= 10) & (use_dof < 30))[0]
    m2_bend_idx = np.where(use_dof >= 30)[0]
    if len(m2_hex_idx) > 0:
        if any(len(block) > 0 for block in [cam_hex_idx, m1m3_bend_idx, m2_bend_idx]):
            ax.axhline(m2_hex_idx[-1] + 0.5, color="k", alpha=0.2)
    if len(cam_hex_idx) > 0:
        if any(len(block) > 0 for block in [m1m3_bend_idx, m2_bend_idx]):
            ax.axhline(cam_hex_idx[-1] + 0.5, color="k", alpha=0.2)
    if len(m1m3_bend_idx) > 0:
        if len(m2_bend_idx) > 0:
            ax.axhline(m1m3_bend_idx[-1] + 0.5, color="k", alpha=0.2)
    ax.set_xticks(range(Vh.shape[1]))
    ax.set_xticklabels(range(1, Vh.shape[1] + 1))
    ax.axvspan(n_keep - 0.5, Vh.shape[1] - 0.5, color="r", alpha=0.2)
    ax.set_yticks(range(Vh.shape[1]))
    ax.set_yticklabels([DOF_NAMES[i] for i in use_dof])
    plt.show()